In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

# 1. Data Preparation
df = pd.read_csv('IMDB Dataset.csv')
# Assume columns are named 'review' (text) and 'sentiment' (e.g., 'positive'/'negative' or 1/0)
X_text = df['review'].astype(str).tolist()
y = df['sentiment'].apply(lambda x: 1 if str(x).lower() in ['positive', '1'] else 0).values

# Tokenize text into lists of words (lowercased)
sentences = [word_tokenize(doc.lower()) for doc in X_text]

# 2. Train Word2Vec CBOW Model (sg=0) to learn word embeddings
w2v_model = Word2Vec(sentences=sentences, vector_size=50, window=5, min_count=1, sg=0, epochs=5)

# 3. Document Representation: Convert each document into a vector by averaging word embeddings
def get_doc_vector(words, model, vector_size=50):
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X_embeddings = np.array([get_doc_vector(doc, w2v_model, vector_size=50) for doc in sentences])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42)

# 4. Model Architecture (Feed-forward neural network for classification)
model = keras.Sequential([
    layers.Input(shape=(50,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary classification (positive/negative)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the classification model
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# 5. Evaluate the Model on test set using loss and accuracy
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")